In [1]:
from src.data_collection.osm_loaders import load_pois


df_osm = load_pois()

/Users/sese/Documents/code/benchmark_pipeline/src/data_collection/osm_extractor.py:130: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


In [2]:
from math import floor
import random

def allocate(n, ratios):          # ratios: {"A": 0.1, "B": 0.3, "C": 0.6}
    total = sum(ratios.values())  # marche aussi avec des poids bruts (1,3,6)
    exact = {k: n * w / total for k, w in ratios.items()}
    counts = {k: floor(v) for k, v in exact.items()}
    for k in sorted(exact, key=lambda k: exact[k] - counts[k], reverse=True)[:n - sum(counts.values())]:
        counts[k] += 1
    return counts

In [ ]:
import random
import math
import networkx as nx
from collections import Counter
from pandas import DataFrame
import numpy as np
from itertools import combinations

from collections import defaultdict
from src.config import *


def generate_question(poi, n_features, features):
    """
    Génère toutes les combinaisons de questions possibles pour un POI donné.

    Évalue chaque feature sur le POI, collecte les fragments de texte disponibles,
    puis construit toutes les combinaisons de `n_features` fragments. Chaque
    combinaison forme une question potentielle de benchmark sous la forme d'une
    liste [catégorie, fragment_1, ..., fragment_n], triée alphabétiquement pour
    garantir la reproductibilité.

    Args:
        poi (Series): Ligne d'un GeoDataFrame représentant un point d'intérêt.
        n_features (int): Nombre de features à combiner dans chaque question.
        features (list[Feature]): Liste des features à évaluer sur le POI.

    Returns:
        list[list[str]] | None: Liste de combinaisons [catégorie, *fragments], ou
            None si le POI ne possède pas assez de features renseignées.
    """
    # Calcule tous les fragments disponibles pour ce POI
    available = []
    for feature in features:
        value = feature.check(poi)
        text = feature.to_text(value)
        if text is not None and text != "nan":
            if isinstance(text, list):
                available.extend(text)  # aplatit les features multi-valeurs
            else:
                available.append(text)

    # Pas assez de features disponibles pour ce POI
    if len(available) < n_features:
        return None
    return [[poi["category"]] + sorted(combo) for combo in combinations(available, n_features)]


def make_questions_semantic(df_osm, features, nb_question=100, ratio=None, rng=None, seed=42):
    if not ratio:
        r = defaultdict(int)
        for k in range(1, len(FEATURES)+1):
            r[k] = 1/len(FEATURES)
        ratio = allocate(nb_question, r)
    if not rng:
        rng = np.random.default_rng(seed)
    category = df_osm["category"].unique()
    dic_question = defaultdict(list)
    for nb_f, nb_q in ratio.items():
        n=0
        while n<nb_q:
            cat_q = rng.choice(category)
            pois = df_osm[df_osm["category"]==cat_q]
            
            id_poi = rng.choice(pois["poi_id"])
            question = generate_question(pois[pois["poi_id"]==id_poi].iloc[0], nb_f, features)
            if question:
                dic_question["question"].append(question[0])
                dic_question["poi"].append(id_poi)
                dic_question["nb_feature"].append(nb_f)
                n += 1
    return DataFrame(dic_question)

In [6]:
q = make_questions_semantic(df_osm, FEATURES)

In [ ]:
def make_benchmark_question(df_osm, distance_max=600, nb_features=2, features=FEATURES):
    """
    fonction qui permet de créer un benchmark de question en choisissant la distance max, c'est à dire la distance à partir 
    de laquelle un poi est considéré comme proche d'une ancre geospatiale.
    """
    dic_question_poi = {}
    dic_poi_question = defaultdict(list)
    for n_features in range(1, nb_features + 1):
        dic_question_poi[n_features] = {}
    for _, poi in df_osm.iterrows():
        for n_features in range(1, nb_features + 1):
            questions_generated = generate_question(poi, n_features, features)
            if not questions_generated:        # gère None ET liste vide
                continue
            for question in questions_generated:
                id_question = " ".join(question)
                if id_question in dic_question_poi[n_features]:   # `in dict` au lieu de `in dict.keys()`
                    dic_question_poi[n_features][id_question].append(poi["index"])
                else:
                    dic_question_poi[n_features][id_question] = [poi["index"]]
                dic_poi_question[poi["index"]].append((n_features, id_question))

    return dict(dic_question_poi), dict(dic_poi_question)


def make_splits(dic_question_poi, feature_coverage, test_ratio=0.1, seed=42):
    """
    feature coverage est le pourcentage de question de chaque catégorie voulut
    feature ratio est le ration de question par feature dans le datset final
    """
    rng = random.Random(seed)
    dataset_size = sum([len(d) for _, d in dic_question_poi.items()])
    train_size = dataset_size * (1 - test_ratio)
    test_size = round(train_size * test_ratio)

    train, test = {}, {}
    feature_ratio = []
    size_dataset = 0
    for n_features, d in dic_question_poi.items():
        keys = list(d)
        rng.shuffle(keys)
        r = feature_coverage[n_features]
        n_train = round(train_size * r)
        n_test = round(test_size * r)
        if n_train + n_test > len(keys):          # pas assez de questions
            n_test = round(len(keys) * test_ratio)
            n_train = round(len(keys) * (1 - test_ratio))
        for q in keys[:n_test]:
            test[q] = (n_features, d[q])
        for q in keys[n_test:n_test + n_train]:    # n_train questions, disjointes du test
            train[q] = (n_features, d[q])
        feature_ratio.append((n_train + n_test))
        size_dataset += (n_train + n_test)
    return train, test, [x / size_dataset for x in feature_ratio]


def build_prefix_index(questions):
    """
    Pré-indexe les questions par préfixe, une seule fois, en O(N).
    Remplace les scans O(N) répétés de get_hard_questions par des lookups O(1).
      - by_template : "chunks[:-1] + chunks[-1][:6]" (strict ET relâché) -> questions
      - by_first    : premier chunk (catégorie)                          -> questions
    """
    by_template = defaultdict(list)
    by_first = defaultdict(list)
    for q in questions:
        chunks = q.split("_")
        by_template["_".join(chunks[:-1] + [chunks[-1][:6]])].append(q)
        by_first[chunks[0]].append(q)
        if len(chunks) >= 2:
            by_template["_".join(chunks[:-2] + [chunks[-1][:6]])].append(q)
    return by_template, by_first


def make_list_batch(train_set, dic_question_poi, dic_question_coords, assignment, size_batch=2048, n_batch=2000, seed=42):
    rng = random.Random(seed)
    list_batch, list_question_training = [], []

    dic_poi_freq = defaultdict(int)

    # tri = ordre déterministe (l'ordre d'un set n'est pas reproductible entre runs),
    # puis un seul shuffle hors boucle au lieu de rng.choice(list(set)) à chaque tour
    list_question_available = sorted(train_set)
    rng.shuffle(list_question_available)

    # index construits une seule fois, partagés par tous les batchs
    by_template, by_first = build_prefix_index(train_set)

    seen = set()   # set au lieu de list : `x not in seen` passe de O(len) à O(1)
    for question in dic_question_coords.keys():
        batch, dic_poi_freq = make_batch2(question, dic_question_poi, dic_question_coords, assignment, dic_poi_freq, size_batch, rng=rng)
        list_batch.append(batch)
        list_question_training.append(question)
        if len(list_batch)>=n_batch:
            return list_batch, list_question_training, dic_poi_freq
    return list_batch, list_question_training, dic_poi_freq

def dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def weighted_choice(poi_list, poi_freq, rng, temperature=100):
    # Inverse la fréquence pour donner moins de poids aux POI fréquents
    weights = [(1.0 / (poi_freq.get(poi, 0) + 1)) ** temperature for poi in poi_list]
    total_weight = sum(weights)
    if total_weight == 0:
        return rng.choice(poi_list)  # fallback
    weights = [w / total_weight for w in weights]
    return rng.choices(poi_list, weights=weights, k=1)[0]

def assign_unique_poi(dic_question_poi):
    """
    AI generated
    """
    questions = list(q for nb_f in dic_question_poi for q in dic_question_poi[nb_f])
    B = nx.Graph()
    q_nodes = [("Q", q) for q in questions]
    B.add_nodes_from(q_nodes, bipartite=0)
    for q in questions:
        nb_f = len(q.split("_"))-1
        for p in dic_question_poi[nb_f][q]:
            B.add_edge(("Q", q), ("P", p))   # tuples : pas de collision nom question / id poi

    matching = nx.bipartite.maximum_matching(B, top_nodes=q_nodes)

    assignment, used = {}, Counter()
    for q in questions:
        node = ("Q", q)
        if node in matching:                  # POI distinct trouvé
            poi = matching[node][1]
            assignment[q] = poi
            used[poi] += 1

    # questions restantes : aucun POI distinct dispo -> on réutilise le moins servi
    for q in questions:
        if q not in assignment:
            nb_f = len(q.split("_"))-1
            pois = dic_question_poi[nb_f][q]
            if pois:
                poi = min(pois, key=lambda p: used[p])
                assignment[q] = poi
                used[poi] += 1
            # si pois == [] : question sans candidat, on la laisse de côté

    return assignment

def make_batch2(question, dic_question_poi, dic_question_coords, assignment, dic_poi_freq, size_batch, rng, distance_min=800):
    ref = dic_question_coords[question]
    chunks = question.split("_")
    nb_feat = len(chunks) - 1
    cat = chunks[0]

    batch = {}
    batch[question] = rng.choice(dic_question_poi[nb_feat][question])

    question_far = [q for q in dic_question_coords
                if q != question and dist(dic_question_coords[q], ref) >= distance_min]
    question_near = [q for q in dic_question_coords
                if q != question and dist(dic_question_coords[q], ref) <= distance_min]  
    question_far = sorted([q for q in question_far if q.split("_")[0] == cat])
    question_near = sorted([q for q in question_near if q.split("_")[0] != cat])
    rng.shuffle(question_far)
    rng.shuffle(question_near)
    poi_answers = []
    for i in range(min(len(question_near), size_batch//3)):
        #chunks = question_near[i].split("_")
        #nb_feat = len(chunks) - 1
        #list_poi = dic_question_poi[nb_feat][question_near[i]]
        #poi = weighted_choice(list_poi, dic_poi_freq, rng)
        #dic_poi_freq[poi] += 1
        batch[question_near[i]] = assignment[question_near[i]]#poi
    
    for i in range(min(len(question_far), (2*size_batch)//3)):
        #chunks = question_far[i].split("_")
        #nb_feat = len(chunks) - 1
        #list_poi = dic_question_poi[nb_feat][question_far[i]]
        #poi = weighted_choice(list_poi, dic_poi_freq, rng)
        #dic_poi_freq[poi] += 1
        poi = assignment[question_far[i]]
        question = question_far[i]
        nb_f = len(question.split("_")) -1
        if poi not in poi_answers:
            batch[question] = assignment[question] #poi
            for p in dic_question_poi[nb_f][question]:
                poi_answers.append(p)
        else:
            continue

    return batch, dic_poi_freq


def make_batch(question, list_question_available, dic_question_poi, size_batch, rng, by_template, by_first, ratio_hard=0.5):
    chunks = question.split("_")
    nb_features = len(chunks) - 1
    seen = set()
    list_hard_question = get_hard_questions(question, size_batch, rng, by_template, by_first, list_question_available)
    pool = dic_question_poi[nb_features]   # référence locale : `in pool` = O(1)
    batch = {}

    for id_hq in range(min(len(list_hard_question), round(size_batch//ratio_hard + size_batch%ratio_hard))):#list_hard_question:
        hard_question = list_hard_question[id_hq]
        if hard_question in pool and hard_question not in seen :
            batch[hard_question] = rng.choice(pool[hard_question])
            seen.add(hard_question)        # dans le if : on ne blackliste que les questions réellement utilisées
        if len(batch) == size_batch:
            return batch
    questions_rdm = rng.sample(list(dic_question_poi[nb_features].keys()), round(size_batch-len(batch)))
    for q in questions_rdm:
        batch[q] = rng.choice(pool[q])

    return batch


def get_hard_questions(question, size_batch, rng, by_template, by_first, list_question_available):
    chunks = question.split("_")

    # scan O(N) du train_set -> lookup O(1) dans l'index préfixe
    question_template = "_".join([c for c in chunks[:-1]] + [chunks[-1][:6]])
    resultats = [s for s in by_template.get(question_template, ()) if s != question and s in list_question_available]

    if len(resultats) == 0 and len(chunks) >= 2:
        question_template = "_".join([c for c in chunks[:-2]] + [chunks[-1][:6]])
        resultats = [s for s in by_template.get(question_template, ()) if s != question]

    if len(resultats) < size_batch:
        # fallback par catégorie via by_first ; le set `deja` évite les doublons resultats/additional
        deja = set(resultats)
        deja.add(question)
        additional_resultats = [s for s in by_first.get(chunks[0], ()) if s not in deja]
        rng.shuffle(additional_resultats)
        resultats = resultats + additional_resultats[:size_batch - len(resultats)]

    rng.shuffle(resultats)
    return resultats
